In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.mart.fct_payments_flat AS
SELECT
  p.payment_id,
  p.order_id,
  p.payment_date,
  p.payment_method,
  p.payment_status,
  p.amount,
  p.is_payment_date_anomaly,

  o.order_date,
  o.order_status,
  o.order_total,
  ROUND(p.amount - o.order_total, 2) AS amount_vs_order_total_diff,

  c.customer_id,
  c.customer_segment,
  c.acquisition_channel,
  c.gender,
  c.city               AS customer_city,

  cust_r.region_id      AS customer_region_id,
  cust_r.region_name    AS customer_region_name,
  cust_r.market_type    AS customer_market_type,

  o.region_id           AS shipping_region_id,
  ship_r.region_name    AS shipping_region_name,
  ship_r.market_type    AS shipping_market_type,

  d.year,
  d.quarter,
  d.month,
  d.month_name,
  d.year_month,
  d.week,
  d.day_name,
  d.is_weekend,
  d.is_holiday,
  d.holiday_name

FROM ecommerce.clean.payments   p
JOIN ecommerce.clean.orders     o        ON p.order_id     = o.order_id
JOIN ecommerce.clean.customers  c        ON o.customer_id  = c.customer_id
LEFT JOIN ecommerce.base.regions cust_r  ON c.region_id     = cust_r.region_id
LEFT JOIN ecommerce.base.regions ship_r  ON o.region_id     = ship_r.region_id
LEFT JOIN ecommerce.base.date   d        ON p.payment_date  = d.date;

### TEST 1 — Row count parity (mart should have exactly one row per payment)

In [0]:
%sql
-- Expect: source_row_count = mart_row_count, row_diff = 0
SELECT
  (SELECT COUNT(*) FROM ecommerce.clean.payments) AS source_row_count,
  (SELECT COUNT(*) FROM ecommerce.mart.fct_payments_flat) AS mart_row_count,
  (SELECT COUNT(*) FROM ecommerce.clean.payments)
    - (SELECT COUNT(*) FROM ecommerce.mart.fct_payments_flat) AS row_diff;

### TEST 2 — Grain check (payment_id must be unique)

In [0]:
%sql
-- Expect: 0 rows
SELECT payment_id, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_payments_flat
GROUP BY payment_id
HAVING COUNT(*) > 1;

### TEST 3 — No fan-out risk from dimension tables (regions, date should each have unique keys)

In [0]:
%sql
-- Expect: 0 rows
-- SELECT 'regions' AS dim, region_id AS key_value, COUNT(*) AS dupes
-- FROM ecommerce.base.regions GROUP BY region_id HAVING COUNT(*) > 1
-- UNION ALL
-- SELECT 'date', date, COUNT(*)
-- FROM ecommerce.base.date GROUP BY date HAVING COUNT(*) > 1;

-- Query is in error state, fixing in progress

### TEST 4 — No orphaned rows dropped by inner joins

In [0]:
%sql
-- Expect: 0 rows
SELECT p.payment_id
FROM ecommerce.clean.payments p
LEFT JOIN ecommerce.mart.fct_payments_flat f ON p.payment_id = f.payment_id
WHERE f.payment_id IS NULL;

### TEST 5 — amount_vs_order_total_diff should be 0 for all payments (dictionary: payments.amount = orders.order_total)

In [0]:
%sql
-- Expect: 0 rows if amount always equals order_total. Any rows here are legitimate
-- data-quality findings for the 'unusual payment patterns' business question — not a build bug,
-- but confirm the count matches what Value_Domain_Checks.ipynb previously found.
SELECT payment_id, order_id, amount, order_total, amount_vs_order_total_diff
FROM ecommerce.mart.fct_payments_flat
WHERE amount_vs_order_total_diff != 0;

### TEST 6 — payment_status distribution + mismatch count (cross-check against original profiling notebook)

In [0]:
%sql
-- Expect: mismatch_count per status should match the counts previously found in
-- Value_Domain_Checks.ipynb (payment_status distribution + amount mismatch check)
SELECT
  payment_status,
  COUNT(*) AS occurrences,
  SUM(CASE WHEN amount_vs_order_total_diff != 0 THEN 1 ELSE 0 END) AS mismatch_count
FROM ecommerce.mart.fct_payments_flat
GROUP BY payment_status
ORDER BY occurrences DESC;

### TEST 7 — is_payment_date_anomaly flag logic (should only be TRUE when payment_date < order_date)

In [0]:
%sql
-- Expect: 0 rows
SELECT payment_id, payment_date, order_date, is_payment_date_anomaly
FROM ecommerce.mart.fct_payments_flat
WHERE (payment_date < order_date AND is_payment_date_anomaly = FALSE)
   OR (payment_date >= order_date AND is_payment_date_anomaly = TRUE);

### TEST 8 — payment_method should be a small, consistent set (cleaning standardized casing/spelling)

In [0]:
%sql
-- Expect: only 'Credit Card', 'Debit Card', 'Digital Wallet', 'Bank Transfer', 'Paypal'
-- (or however INITCAP renders PayPal) -- no lowercase/mixed variants
SELECT payment_method, COUNT(*) AS occurrences
FROM ecommerce.mart.fct_payments_flat
GROUP BY payment_method
ORDER BY occurrences DESC;

### TEST 9 — Null profiling on key columns

In [0]:
%sql
-- Expect: 0 across all columns listed here
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) - COUNT(payment_method)       AS null_payment_method,
  COUNT(*) - COUNT(payment_status)       AS null_payment_status,
  COUNT(*) - COUNT(amount)               AS null_amount,
  COUNT(*) - COUNT(order_total)          AS null_order_total,
  COUNT(*) - COUNT(customer_segment)     AS null_segment,
  COUNT(*) - COUNT(year)                 AS null_date_join
FROM ecommerce.mart.fct_payments_flat;

### TEST 10 — Referential sanity (order_id and customer_id resolve to clean dimensions)

In [0]:
%sql
-- Expect: 0 rows for each check_type
SELECT 'order_id' AS check_type, f.order_id AS bad_key
FROM ecommerce.mart.fct_payments_flat f
LEFT JOIN ecommerce.clean.orders o ON f.order_id = o.order_id
WHERE o.order_id IS NULL
UNION ALL
SELECT 'customer_id', f.customer_id
FROM ecommerce.mart.fct_payments_flat f
LEFT JOIN ecommerce.clean.customers c ON f.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

### TEST 11 — Value range sanity (amount and order_total should never be negative or zero)

In [0]:
%sql
-- Expect: 0 rows
SELECT *
FROM ecommerce.mart.fct_payments_flat
WHERE amount <= 0
   OR order_total <= 0;

### TEST 12 — Date join sanity (payment_date should match the joined date-dimension row exactly)

In [0]:
%sql
-- Expect: 0 rows
SELECT f.payment_id, f.payment_date, d.date AS joined_date
FROM ecommerce.mart.fct_payments_flat f
LEFT JOIN ecommerce.base.date d ON f.payment_date = d.date
WHERE d.date IS NULL OR f.payment_date != d.date;

### TEST 13 — Excluded fields truly absent (item/return-grain fields should NOT be in this table)

In [0]:
%sql
-- Expect: query FAILS (column not found) — that failure is the pass condition
SELECT order_item_id, product_id, return_id, refund_amount FROM ecommerce.mart.fct_payments_flat LIMIT 1;

### TEST 14 — Distinct value spot-check on payment_status

In [0]:
%sql
-- Expect: only 'Successful', 'Failed', 'Refunded'
SELECT DISTINCT payment_status FROM ecommerce.mart.fct_payments_flat;

In [0]:
%sql
-- Check 1: Are payment_dates falling outside the date dimension's known range?
SELECT
  MIN(payment_date) AS min_payment_date,
  MAX(payment_date) AS max_payment_date,
  (SELECT MIN(date) FROM ecommerce.base.date) AS dim_min_date,
  (SELECT MAX(date) FROM ecommerce.base.date) AS dim_max_date
FROM ecommerce.clean.payments;

In [0]:
%sql
-- Check 2: How many payment_dates are literally NULL?
SELECT COUNT(*) AS null_payment_dates
FROM ecommerce.clean.payments
WHERE payment_date IS NULL;

In [0]:
%sql
-- Check 3: Data type / time-component check — cast payment_date to DATE and re-test the join
SELECT COUNT(*) AS still_unmatched
FROM ecommerce.clean.payments p
LEFT JOIN ecommerce.base.date d ON CAST(p.payment_date AS DATE) = d.date
WHERE d.date IS NULL;

In [0]:
%sql
-- Check 4: Distribution of the unmatched payment_dates by year — tells you if it's a range issue
SELECT YEAR(payment_date) AS pay_year, COUNT(*) AS occurrences
FROM ecommerce.clean.payments p
WHERE NOT EXISTS (SELECT 1 FROM ecommerce.base.date d WHERE d.date = p.payment_date)
GROUP BY YEAR(payment_date)
ORDER BY pay_year;

In [0]:
%sql
SELECT MAX(return_date) AS max_return_date
FROM ecommerce.clean.returns;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.base.date AS
SELECT * FROM ecommerce.base.date  -- existing 2023-2025 rows, untouched
UNION ALL
SELECT
  d.date_val                                     AS date,
  DAY(d.date_val)                                AS day,
  DATE_FORMAT(d.date_val, 'EEEE')                AS day_name,
  WEEKOFYEAR(d.date_val)                         AS week,
  MONTH(d.date_val)                              AS month,
  DATE_FORMAT(d.date_val, 'MMMM')                AS month_name,
  QUARTER(d.date_val)                            AS quarter,
  YEAR(d.date_val)                               AS year,
  DATE_FORMAT(d.date_val, 'yyyy-MM')             AS year_month,
  CASE WHEN DAYOFWEEK(d.date_val) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
  FALSE                                           AS is_holiday,   -- no holiday calendar defined for extension period
  NULL                                            AS holiday_name
FROM (
  SELECT EXPLODE(SEQUENCE(DATE '2026-01-01', DATE '2026-03-31', INTERVAL 1 DAY)) AS date_val
) d;